In [9]:
import pandas as pd 
import numpy as np


In [10]:
df = pd.read_csv("benchmarking_predictions.tsv", sep = "\t")
AMINO_ACIDS = list("ARNDCQEGHILKMFPSTWYV")
POSITIONS = [-13, -12, -11, -10, -9, -8, -7, -6, -5, -4, -3, -2, -1, 1, 2]
MODELS = {"von Heijne": "pred_von_heijne", "SVM": "pred_svm"}
df.head()

,accession,label,organism,kingdom,length,cleavage_site,tm_in_first_90,sequence,vh_score,pred_von_heijne,pred_svm
0,Q8MMH3,1,Pimpla hypochondriaca (Parasitoid wasp),Metazoa,488,19.0,NaN,MWSPLILPLLAQIFACAFGDAVNEGCNVDPKGIHGLWYLDEPFGVY...,6.214328,0,1
1,Q6TPK4,1,Actinidia deliciosa (Kiwi),Plants,116,26.0,NaN,MVPKPLSLLLFLLLALSAAVVGGRKLVAAGGWRPIESLNSAEVQDV...,10.603172,1,1
2,Q62632,1,Rattus norvegicus (Rat),Metazoa,306,18.0,NaN,MWKRWLALALVTIALVHGEEEQRSKSKICANVFCGAGRECAVTEKG...,9.458926,1,1
3,P47710,1,Homo sapiens (Human),Metazoa,185,15.0,NaN,MRLLILTCLVAVALARPKLPLRYPERLQNPSESSEPIPLESREEYM...,12.218489,1,1
4,Q8TAL6,1,Homo sapiens (Human),Metazoa,211,18.0,NaN,MVFLKFFCMSFFCHLCQGYFDGPLYPEMSNGTLHHYFVPDGDYEEN...,8.166644,1,0


In [16]:
for name, col in MODELS.items():
    print(f"\n--- Error distribution among taxa: {name} ---")
    summary = df.groupby("kingdom").apply(lambda g: pd.Series({
        "Total": len(g),
        "Positives (SP)": (g["label"] == 1).sum(),
        "Negatives (non-SP)": (g["label"] == 0).sum(),
        "FP": ((g["label"] == 0) & (g[col] == 1)).sum(),
        "FN": ((g["label"] == 1) & (g[col] == 0)).sum(),
    }), include_groups=False).astype(int)

    summary["FPR (%)"] = (summary["FP"] / summary["Negatives (non-SP)"] * 100).round(2)
    summary["FNR (%)"] = (summary["FN"] / summary["Positives (SP)"] * 100).round(2)
    print(summary.to_string())


--- Error distribution among taxa: von Heijne ---
         Total  Positives (SP)  Negatives (non-SP)  FP  FN  FPR (%)  FNR (%)
kingdom                                                                     
Fungi      544              22                 522   5   7     0.96    31.82
Metazoa   1118             179                 939  48  45     5.11    25.14
Other       44               1                  43   4   0     9.30     0.00
Plants     330              18                 312  18   3     5.77    16.67

--- Error distribution among taxa: SVM ---
         Total  Positives (SP)  Negatives (non-SP)  FP  FN  FPR (%)  FNR (%)
kingdom                                                                     
Fungi      544              22                 522   3   8     0.57    36.36
Metazoa   1118             179                 939  30  21     3.19    11.73
Other       44               1                  43   1   1     2.33   100.00
Plants     330              18                 312   4   1

In [17]:
print("2. FALSE POSITIVE ANALYSIS: TRANSMEMBRANE (TM) HELICES")

print("\n--- FP with TM helix within first 90 residues ---")
for name, col in MODELS.items():
    fp_mask = (df["label"] == 0) & (df[col] == 1)
    tot_fp = fp_mask.sum()
    tm_fp = (fp_mask & (df["tm_in_first_90"] == True)).sum()
    perc = (tm_fp / tot_fp * 100) if tot_fp > 0 else 0.0
    print(f"{name:10s} -> Total FP: {tot_fp:3d} | with TM: {tm_fp:3d} ({perc:.2f}%)")

2. FALSE POSITIVE ANALYSIS: TRANSMEMBRANE (TM) HELICES

--- FP with TM helix within first 90 residues ---
von Heijne -> Total FP:  75 | with TM:  41 (54.67%)
SVM        -> Total FP:  38 | with TM:  23 (60.53%)


In [ ]:
print("3. FALSE NEGATIVE ANALYSIS (VON HEIJNE)")

def extract_context(row):
    if pd.isna(row["cleavage_site"]):
        return None
    c = int(row["cleavage_site"])
    seq = str(row["sequence"])
    start, end = c - 13, c + 2
    if start >= 0 and end <= len(seq):
        ctx = seq[start:end]
        return ctx if len(ctx) == len(POSITIONS) else None
    return None

df["cleavage_context"] = df.apply(extract_context, axis=1)

tp_vh = df[(df["label"] == 1) & (df["pred_von_heijne"] == 1)].dropna(subset=["cleavage_context"])
fn_vh = df[(df["label"] == 1) & (df["pred_von_heijne"] == 0)].dropna(subset=["cleavage_context"])

#compute amminoacidic frequency matrix for all 15 position (of the cleavage context) for both TP and FN
def get_position_frequencies(series_contexts):
    matrix = pd.DataFrame(0.0, index = POSITIONS, columns = AMINO_ACIDS)
    for pos_idx, pos_val in enumerate(POSITIONS):
        counts = series_contexts.str[pos_idx].value_counts(normalize = True) * 100
        for aa, val in counts.items():
            if aa in matrix.columns:
                matrix.at[pos_val, aa] = val
    return matrix.round(2)

tp_freq_matrix = get_position_frequencies(tp_vh["cleavage_context"])
fn_freq_matrix = get_position_frequencies(fn_vh["cleavage_context"])
diff_freq_matrix = (fn_freq_matrix - tp_freq_matrix).round(2)

print("\n--- Differential aa frequency (FN - TP) in the cleavage site (15 positions x 20 AA) ---")
print(diff_freq_matrix.to_string())
print("\n--- Negative values---> abundant aa in TP wrt FN")
print("\n--- Positive values---> abundant aa in FN wrt TP")

3. FALSE NEGATIVE ANALYSIS (VON HEIJNE)

--- Differential aa frequency (FN - TP) in the cleavage site (15 positions x 20 AA) ---
         A     R     N     D      C     Q     E     G     H      I      L     K     M      F     P     S     T     W     Y      V
-13   1.82 -0.61  0.00 -0.61  -4.85 -0.61  0.00  0.61  0.00   3.03 -34.54 -0.61  5.45   1.21  2.43  6.06  2.43  4.85  3.03  10.91
-12  -3.64  0.00  0.00 -0.61  -0.60  1.82  0.00  3.63  1.82   9.69 -16.97  0.00  4.24  -6.06  1.82  1.81  3.03 -0.61  1.82  -1.21
-11  -1.21  0.00  0.00  0.00  -3.03  1.21 -0.61  0.00  1.82  -4.25 -20.00  0.00  4.85   2.42  1.21  9.09  3.63 -0.61  3.03   2.43
-10  -1.21 -0.61  1.82  0.00   2.42  0.00  0.00 -4.85 -0.61   5.45  -9.09  0.00 -1.21  10.30  1.22 -4.84  0.61 -0.60  0.00   1.21
-9   -9.09  1.82  1.82 -0.61  -3.03  1.21  1.21 -4.24 -0.61  11.52 -13.34  0.00  5.45   6.06  3.63 -4.84  3.03 -1.21 -0.61   1.82
-8   -8.48  0.00  1.21 -0.61 -10.30  6.66  0.00  0.00  1.82  10.31 -12.73  0.00  6.67   8.4

In [ ]:
print("4. DIAGNOSING SVM FALSE NEGATIVES: LENGTH & N-TERMINAL COMPOSITION")


tp_svm = df[(df["label"] == 1) & (df["pred_svm"] == 1)]
fn_svm = df[(df["label"] == 1) & (df["pred_svm"] == 0)]

len_stats = pd.DataFrame({
    "Benchmark TP": tp_svm["cleavage_site"].describe(),
    "Benchmark FN": fn_svm["cleavage_site"].describe()
}).round(2)
print("\n--- SP length stats (SVM TP vs FN) ---")
print(len_stats.to_string())

# first n residues aa composition
n = int(round(df.query("label == 1")["cleavage_site"].median()))

all_aa_data = []
for aa in AMINO_ACIDS:
    tp_f = tp_svm["sequence"].str[:n].apply(lambda s: s.count(aa) / len(s) if len(s) > 0 else 0).mean() * 100
    fn_f = fn_svm["sequence"].str[:n].apply(lambda s: s.count(aa) / len(s) if len(s) > 0 else 0).mean() * 100
    all_aa_data.append({
        "Residue": aa,
        "TP Mean (%)": round(tp_f, 2),
        "FN Mean (%)": round(fn_f, 2),
        "Diff (FN - TP) (%)": round(fn_f - tp_f, 2)
    })

svm_aa_comp_df = pd.DataFrame(all_aa_data).sort_values(by = "Residue").reset_index(drop = True)
print(f"\n--- AA frequency within first {n} residues (SVM TP vs FN) ---")
print(svm_aa_comp_df.to_string(index = False))

4. DIAGNOSING SVM FALSE NEGATIVES: LENGTH & N-TERMINAL COMPOSITION

--- SP length stats (SVM TP vs FN) ---
       Benchmark TP  Benchmark FN
count        189.00         31.00
mean          22.10         25.52
std            4.24          9.62
min           15.00         17.00
25%           19.00         19.00
50%           22.00         24.00
75%           25.00         28.50
max           33.00         64.00

--- AA frequency within first 22 residues (SVM TP vs FN) ---
Residue  TP Mean (%)  FN Mean (%)  Diff (FN - TP) (%)
      A        13.73         9.38               -4.35
      C         3.13         2.35               -0.78
      D         1.25         1.32                0.07
      E         1.54         1.32               -0.22
      F         5.29         5.87                0.57
      G         4.52         8.36                3.84
      H         1.13         2.05                0.92
      I         5.05         5.13                0.08
      K         2.19         0.88      

In [ ]:

df = pd.read_csv("benchmarking_predictions.tsv", sep = "\t")

# only FN after VON HRJNE inference
fn_vh = df[(df["label"] == 1) & (df["pred_von_heijne"] == 0)].copy()

def extract_context(row):
    c = int(row["cleavage_site"])
    seq = str(row["sequence"])
    start, end = c - 13, c + 2
    if start >= 0 and end <= len(seq):
        ctx = seq[start:end]
        return ctx if len(ctx) == 15 else None
    return None

fn_vh["context"] = fn_vh.apply(extract_context, axis = 1) #extracting cleavage site from FN only
fn_vh = fn_vh.dropna(subset = ["context"])

# saving contexts in a FAST file to compute the sequence LOGO
with open("fn_vh_contexts.fasta", "w") as f:
    for _, r in fn_vh.iterrows():
        f.write(f">{r['accession']}|site_{int(r['cleavage_site'])}\n{r['context']}\n")
